In [12]:
# HW0: robust mega acquisition (Colab-first, git-safe)
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = "https://github.com/egil10/stk-mat2011.git"
REPO_DIR = Path("/content/stk-mat2011") if IN_COLAB else Path.cwd().resolve().parents[1]

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        # keep notebook reproducible if you re-run later
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)

    # optional but useful when Colab VM is fresh
    req = REPO_DIR / "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

# make scripts importable
SCRIPTS_DIR = REPO_DIR / "code" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

from p_duka import download_and_save

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_DIR={REPO_DIR}")
print(f"SCRIPTS_DIR={SCRIPTS_DIR}")

IN_COLAB=True
REPO_DIR=/content/stk-mat2011
SCRIPTS_DIR=/content/stk-mat2011/code/scripts


In [13]:
from datetime import datetime
import shutil
import pandas as pd

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

# Repo output path (where p_duka writes)
repo_processed = REPO_DIR / "code" / "data" / "processed"
repo_processed.mkdir(parents=True, exist_ok=True)

# Persistent cache on Drive (survives git pull/reclone)
drive_processed = Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed") if IN_COLAB else repo_processed
if IN_COLAB:
    drive_processed.mkdir(parents=True, exist_ok=True)


def month_list(start_ym: str, end_ym: str):
    """Inclusive YYYYMM range."""
    y0, m0 = int(start_ym[:4]), int(start_ym[4:6])
    y1, m1 = int(end_ym[:4]), int(end_ym[4:6])
    out = []
    y, m = y0, m0
    while (y < y1) or (y == y1 and m <= m1):
        out.append(f"{y}{m:02d}")
        if m == 12:
            y, m = y + 1, 1
        else:
            m += 1
    return out


def expected_files(pair: str, yyyymm: str):
    s = pair.replace("/", "").lower()
    return [
        f"{s}_dukascopy_bid_{yyyymm}.parquet",
        f"{s}_dukascopy_ask_{yyyymm}.parquet",
    ]


def sync_drive_to_repo():
    if not IN_COLAB:
        return
    copied = 0
    for fp in drive_processed.glob("*.parquet"):
        tgt = repo_processed / fp.name
        if not tgt.exists():
            shutil.copy2(fp, tgt)
            copied += 1
    print(f"Synced Drive -> repo: {copied} new file(s)")


def sync_repo_to_drive():
    if not IN_COLAB:
        return
    copied = 0
    for fp in repo_processed.glob("*.parquet"):
        tgt = drive_processed / fp.name
        if not tgt.exists():
            shutil.copy2(fp, tgt)
            copied += 1
    print(f"Synced repo -> Drive: {copied} new file(s)")


print(f"repo_processed={repo_processed}")
print(f"drive_processed={drive_processed}")

Mounted at /content/drive
repo_processed=/content/stk-mat2011/code/data/processed
drive_processed=/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed


In [ ]:
# Fast acquisition: newest-first + chunked + periodic sync
PAIR = "EUR/USD"

# 1) Choose one chunk per run (recommended)
# Example quick chunk:
START_YYYYMM = "202301"
END_YYYYMM = "202507"
SYNC_EVERY = 1
MAX_FAIL_BEFORE_STOP = 3

# Pull persistent files into repo first
sync_drive_to_repo()

months = month_list(START_YYYYMM, END_YYYYMM)
months = list(reversed(months))  # newest first
print(f"Planned months: {len(months)} ({months[-1]} -> {months[0]}) newest-first")

ok, skip, fail = [], [], []
ok_since_sync = 0

for i, yyyymm in enumerate(months, 1):
    names = expected_files(PAIR, yyyymm)
    have = all((drive_processed / n).exists() or (repo_processed / n).exists() for n in names)

    if have:
        skip.append(yyyymm)
        if i % 10 == 0:
            print(f"[{i}/{len(months)}] skip {yyyymm} (already exists)")
        continue

    print(f"[{i}/{len(months)}] fetch {PAIR} {yyyymm}")
    try:
        _ = download_and_save(PAIR, yyyymm, compression="snappy")
        ok.append(yyyymm)
        ok_since_sync += 1

        # periodic sync to persistent storage
        if ok_since_sync >= SYNC_EVERY:
            sync_repo_to_drive()
            ok_since_sync = 0

    except Exception as e:
        print(f"  ERROR {yyyymm}: {e}")
        fail.append(yyyymm)

        # safety stop if endpoint is flaky
        if len(fail) >= MAX_FAIL_BEFORE_STOP:
            print(f"Stopping early: reached {MAX_FAIL_BEFORE_STOP} failures.")
            break

# final sync
sync_repo_to_drive()

print("\nDone")
print(f"Downloaded: {len(ok)}")
print(f"Skipped existing: {len(skip)}")
print(f"Failed: {len(fail)}")
if ok:
    print("Newest downloaded months:", ok[:10])   # because newest-first
if fail:
    print("Failed months:", fail[:20])

Synced Drive -> repo: 0 new file(s)
Planned months: 31 (202301 -> 202507) newest-first
[1/31] fetch EUR/USD 202507


INFO:DUKASCRIPT:current timestamp :2025-07-01T09:20:21.098000
INFO:DUKASCRIPT:current timestamp :2025-07-01T15:22:49.320000
INFO:DUKASCRIPT:current timestamp :2025-07-02T06:20:16.625000
INFO:DUKASCRIPT:current timestamp :2025-07-02T13:07:45.501000
INFO:DUKASCRIPT:current timestamp :2025-07-03T00:30:36.293000
INFO:DUKASCRIPT:current timestamp :2025-07-03T12:13:02.565000
INFO:DUKASCRIPT:current timestamp :2025-07-03T15:38:06.655000
INFO:DUKASCRIPT:current timestamp :2025-07-04T07:59:06.621000
INFO:DUKASCRIPT:current timestamp :2025-07-04T19:29:33.450000
INFO:DUKASCRIPT:current timestamp :2025-07-07T07:41:18.130000
INFO:DUKASCRIPT:current timestamp :2025-07-07T16:53:22.125000
INFO:DUKASCRIPT:current timestamp :2025-07-08T04:16:00.842000
INFO:DUKASCRIPT:current timestamp :2025-07-08T13:48:23.268000
INFO:DUKASCRIPT:current timestamp :2025-07-09T00:28:22.005000
INFO:DUKASCRIPT:current timestamp :2025-07-09T10:20:15.314000
INFO:DUKASCRIPT:current timestamp :2025-07-09T22:19:53.120000
INFO:DUK

  Received 1,558,153 ticks
Synced repo -> Drive: 2 new file(s)
[2/31] fetch EUR/USD 202506


INFO:DUKASCRIPT:current timestamp :2025-06-02T07:03:53.384000
INFO:DUKASCRIPT:current timestamp :2025-06-02T12:06:41.159000
INFO:DUKASCRIPT:current timestamp :2025-06-02T16:28:06.656000
INFO:DUKASCRIPT:current timestamp :2025-06-03T02:52:00.673000
INFO:DUKASCRIPT:current timestamp :2025-06-03T09:41:46.536000
INFO:DUKASCRIPT:current timestamp :2025-06-03T14:41:33.746000
INFO:DUKASCRIPT:current timestamp :2025-06-04T01:32:19.677000
INFO:DUKASCRIPT:current timestamp :2025-06-04T08:13:41.421000
INFO:DUKASCRIPT:current timestamp :2025-06-04T13:55:10.303000
INFO:DUKASCRIPT:current timestamp :2025-06-04T22:18:38.030000
INFO:DUKASCRIPT:current timestamp :2025-06-05T07:33:12.413000
INFO:DUKASCRIPT:current timestamp :2025-06-05T13:00:11.290000
INFO:DUKASCRIPT:current timestamp :2025-06-05T15:16:20.204000
INFO:DUKASCRIPT:current timestamp :2025-06-06T01:47:42.820000
INFO:DUKASCRIPT:current timestamp :2025-06-06T10:42:15.378000
INFO:DUKASCRIPT:current timestamp :2025-06-06T14:02:18.720000
INFO:DUK

  Received 2,033,530 ticks
Synced repo -> Drive: 2 new file(s)
[3/31] fetch EUR/USD 202505


...

INFO:DUKASCRIPT:current timestamp :2024-11-01T11:14:33.987000
INFO:DUKASCRIPT:current timestamp :2024-11-01T14:44:15.896000
INFO:DUKASCRIPT:current timestamp :2024-11-04T01:02:48.980000
INFO:DUKASCRIPT:current timestamp :2024-11-04T09:28:49.513000
INFO:DUKASCRIPT:current timestamp :2024-11-04T16:31:10.965000
INFO:DUKASCRIPT:current timestamp :2024-11-05T08:18:21.572000
INFO:DUKASCRIPT:current timestamp :2024-11-05T16:29:51.790000
INFO:DUKASCRIPT:current timestamp :2024-11-06T00:41:20.120000
INFO:DUKASCRIPT:current timestamp :2024-11-06T02:32:16.513000
INFO:DUKASCRIPT:current timestamp :2024-11-06T04:16:52.535000
INFO:DUKASCRIPT:current timestamp :2024-11-06T06:30:00.210000
INFO:DUKASCRIPT:current timestamp :2024-11-06T08:32:59.823000
INFO:DUKASCRIPT:current timestamp :2024-11-06T11:41:28.063000
INFO:DUKASCRIPT:current timestamp :2024-11-06T14:14:47.721000
INFO:DUKASCRIPT:current timestamp :2024-11-06T17:06:36.345000
INFO:DUKASCRIPT:current timestamp :2024-11-07T01:18:34.390000
INFO:DUK

  Received 2,323,551 ticks
Synced repo -> Drive: 2 new file(s)
[10/31] fetch EUR/USD 202410


INFO:DUKASCRIPT:current timestamp :2024-10-01T09:21:17.758000
INFO:DUKASCRIPT:current timestamp :2024-10-01T14:02:07.515000
INFO:DUKASCRIPT:current timestamp :2024-10-01T17:54:07.230000
INFO:DUKASCRIPT:current timestamp :2024-10-02T06:48:35.771000
INFO:DUKASCRIPT:current timestamp :2024-10-02T13:22:36.027000
INFO:DUKASCRIPT:current timestamp :2024-10-02T20:55:13.956000
INFO:DUKASCRIPT:current timestamp :2024-10-03T08:27:14.414000
INFO:DUKASCRIPT:current timestamp :2024-10-03T13:59:23.904000
INFO:DUKASCRIPT:current timestamp :2024-10-03T18:20:30.593000
INFO:DUKASCRIPT:current timestamp :2024-10-04T07:50:32.393000
INFO:DUKASCRIPT:current timestamp :2024-10-04T13:03:10.467000
INFO:DUKASCRIPT:current timestamp :2024-10-04T17:27:36.103000
INFO:DUKASCRIPT:current timestamp :2024-10-07T06:43:30.175000
INFO:DUKASCRIPT:current timestamp :2024-10-07T12:38:00.215000
INFO:DUKASCRIPT:current timestamp :2024-10-07T19:07:22.651000
INFO:DUKASCRIPT:current timestamp :2024-10-08T06:31:35.642000
INFO:DUK

  Received 1,769,081 ticks
Synced repo -> Drive: 2 new file(s)
[11/31] fetch EUR/USD 202409


INFO:DUKASCRIPT:current timestamp :2024-09-02T10:39:36.413000
INFO:DUKASCRIPT:current timestamp :2024-09-03T05:43:29.046000
INFO:DUKASCRIPT:current timestamp :2024-09-03T12:47:56.129000
INFO:DUKASCRIPT:current timestamp :2024-09-03T16:54:36.003000
INFO:DUKASCRIPT:current timestamp :2024-09-04T07:13:49.983000
INFO:DUKASCRIPT:current timestamp :2024-09-04T14:09:43.586000
INFO:DUKASCRIPT:current timestamp :2024-09-05T01:33:06.163000
INFO:DUKASCRIPT:current timestamp :2024-09-05T12:08:24.790000
INFO:DUKASCRIPT:current timestamp :2024-09-05T15:28:21.456000
INFO:DUKASCRIPT:current timestamp :2024-09-06T07:44:34.704000
INFO:DUKASCRIPT:current timestamp :2024-09-06T13:04:50.788000
INFO:DUKASCRIPT:current timestamp :2024-09-06T15:02:25.122000
INFO:DUKASCRIPT:current timestamp :2024-09-06T19:06:48.258000
INFO:DUKASCRIPT:current timestamp :2024-09-09T08:28:25.317000
INFO:DUKASCRIPT:current timestamp :2024-09-09T15:22:50.244000
INFO:DUKASCRIPT:current timestamp :2024-09-10T07:35:35.690000
INFO:DUK

In [ ]:
# Manifest / health check
store = drive_processed if IN_COLAB else repo_processed
files = sorted(store.glob("*.parquet"))
print(f"Store: {store}")
print(f"Total parquet files: {len(files)}")

total_mb = sum(f.stat().st_size for f in files) / (1024 * 1024)
print(f"Total size: {total_mb:.1f} MB")

rows = []
for fp in files:
    name = fp.name
    # expected format: eurusd_dukascopy_bid_202101.parquet
    parts = name.replace(".parquet", "").split("_")
    if len(parts) >= 4:
        yyyymm = parts[-1]
        side = parts[-2]
    else:
        yyyymm = ""
        side = ""
    rows.append({"file": name, "yyyymm": yyyymm, "side": side, "mb": fp.stat().st_size / 1e6})

m = pd.DataFrame(rows)
if len(m):
    print("\nFiles by month (head):")
    display(m.sort_values(["yyyymm", "side"]).head(20))

    month_counts = m.groupby("yyyymm").size().rename("n_files").reset_index().sort_values("yyyymm")
    print("\nMonth coverage (last 24):")
    display(month_counts.tail(24))

# Note for gitignore/git pull workflow:
# - Keep parquet in Drive path above (not in git).
# - Notebook syncs between repo output and Drive cache.
# - After git pull/reclone, data remains in Drive and is re-synced.

Store: /content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed
Total parquet files: 22
Total size: 284.9 MB

Files by month (head):


,file,yyyymm,side,mb
0,eurusd_dukascopy_ask_202101.parquet,202101,ask,18.387604
11,eurusd_dukascopy_bid_202101.parquet,202101,bid,18.384328
1,eurusd_dukascopy_ask_202102.parquet,202102,ask,15.310751
12,eurusd_dukascopy_bid_202102.parquet,202102,bid,15.227004
2,eurusd_dukascopy_ask_202508.parquet,202508,ask,12.352547
13,eurusd_dukascopy_bid_202508.parquet,202508,bid,12.384845
3,eurusd_dukascopy_ask_202509.parquet,202509,ask,13.929453
14,eurusd_dukascopy_bid_202509.parquet,202509,bid,14.017766
4,eurusd_dukascopy_ask_202510.parquet,202510,ask,13.901355
15,eurusd_dukascopy_bid_202510.parquet,202510,bid,13.918950



Month coverage (last 24):


,yyyymm,n_files
0,202101,2
1,202102,2
2,202508,2
3,202509,2
4,202510,2
5,202511,2
6,202512,2
7,202601,2
8,202602,2
9,202603,2
